In [1]:
import cracknuts as cn

In [2]:
cracker = cn.new_cracker('192.168.0.180')

C:\Users\Dingzb\AppData\Local\Temp\ipykernel_8480\2095376612.py:1: DeprecationWarning: new_cracker() is deprecated and has been disabled. Use cracker_s1(...) or cracker_g1(...) instead.
  cracker = cn.new_cracker('192.168.0.180')


In [3]:
cracker.connect()

In [14]:
import random
import time
from cracknuts.cracker import serial

# from Crypto.Cipher import AES 


cmd_set_aes_enc_key = "01 00 00 00 00 00 00 10"
cmd_aes_enc = "01 02 00 00 00 00 00 10"

aes_key = "11 22 33 44 55 66 77 88 99 00 aa bb cc dd ee ff"
aes_data_len = 16

sample_length = 300


# cipher = AES.new(bytes.fromhex(aes_key), AES.MODE_ECB) 
    

def init(c):
    cracker.nut_clock_freq('24M')
    cracker.nut_clock_enable()
    time.sleep(0.1)
    cracker.nut_voltage(3.3)
    cracker.nut_voltage_enable()

    cracker.osc_sample_clock('48m')
    cracker.osc_sample_length(sample_length)
    cracker.osc_trigger_source('N')
    cracker.osc_analog_gain('B', 34)
    cracker.osc_trigger_level(0)
    cracker.osc_trigger_mode('E')
    cracker.osc_trigger_edge('U')

    time.sleep(0.2)
    print("disabling uart")
    cracker.uart_io_disable()
    cracker.uart_reset()
    time.sleep(0.1)
    print("enabling uart")
    cracker.uart_config(baudrate=serial.Baudrate.BAUDRATE_9600, bytesize=serial.Bytesize.EIGHTBITS, parity=serial.Parity.PARITY_NONE, stopbits=serial.Stopbits.STOPBITS_ONE)
    cracker.uart_io_enable()
    time.sleep(1)
    print("setting key")
    cmd = cmd_set_aes_enc_key + aes_key
    status, ret = cracker.uart_transmit_receive(cmd, timeout=1000, rx_count=6)
    time.sleep(1)
    print("dummy writing")
    plaintext_data = random.randbytes(aes_data_len)
    tx_data = bytes.fromhex(cmd_aes_enc) + plaintext_data
    status, ret = cracker.uart_transmit_receive(tx_data, rx_count= 6 + aes_data_len, is_trigger=False)
    time.sleep(0.2)
    print("running...")

def do(c, cc):
    plaintext_data = random.randbytes(aes_data_len) 
    # plaintext_data = "cracknuts0123456".encode('utf-8')
    tx_data = bytes.fromhex(cmd_aes_enc) + plaintext_data
    status, ret = cracker.uart_transmit_receive(tx_data, rx_count= 6 + aes_data_len, is_trigger=True)

    # try:
    #     right_ct = cipher.encrypt(plaintext_data)
    # except: 
    #     print("encrypt error")

    # try: 
    #     print(right_ct.hex(), "||", ret[:].hex()) 
    #     print("hello")
    # except:
    #     print("print error")
    
    return {
        "plaintext": plaintext_data,
        "ciphertext": ret[-aes_data_len:],
        "key": bytes.fromhex(aes_key)
    }

def finish(c):
    cracker.nut_clock_disable()
    cracker.nut_voltage_disable()

acq = cn.simple_acq(cracker, do_func=do, init_func=init, finish_func=finish)

In [15]:
cn.show_panel(acq)

CracknutsPanelWidget(acq_run_progress={'finished': 0, 'total': -1}, connect_status=True, custom_y_range={'0': …

disabling uart
enabling uart
setting key
dummy writing
running...
disabling uart
enabling uart
setting key
dummy writing
running...
disabling uart
enabling uart
setting key
dummy writing
running...
